# 01 - Metodologia do dataset e splits

Este notebook documenta a base metodologica do projeto: estrutura do dataset, conceito de slice, agrupamento por `inferred_group_id` e validacao do split por grupo.

O objetivo e deixar claro para leitores medicos e academicos que a unidade visual de entrada e o slice JPEG, mas a separacao entre treino, validacao e teste deve respeitar o agrupamento inferido para reduzir risco de vazamento.

Este estudo e exploratorio. O campo `inferred_group_id` e tecnico, reconstruido a partir do nome do arquivo, e nao deve ser tratado como identificador clinico validado de paciente ou exame.


## tl;dr

- O problema e uma classificacao binaria entre `Healthy` e `Hepatic_Steatosis`.
- Cada arquivo JPEG representa um slice/corte de tomografia exportado como imagem 2D.
- Varios slices podem pertencer ao mesmo agrupamento inferido pelo nome do arquivo.
- O split deve ser feito por `inferred_group_id`, nunca por slice individual.
- A principal checagem metodologica deste notebook e confirmar que nenhum `inferred_group_id` aparece em mais de um split.


## Contexto & Metodos

As imagens reais nao sao versionadas no repositorio. Os arquivos em `data/interim/` sao derivados locais criados pelos scripts do projeto:

```bash
python scripts/build_dataset_index.py
python scripts/build_splits.py
```

Arquivos esperados nesta etapa:

- `data/interim/dataset_index.csv`: uma linha por imagem/slice.
- `data/interim/group_index.csv`: uma linha por `inferred_group_id`.
- `data/interim/split_groups.csv`: grupos inferidos com coluna `split`.
- `data/interim/split_slices.csv`: slices com split herdado do grupo.

### Key Assumptions

- As classes sao derivadas da estrutura de pastas do dataset.
- O dataset local esta em JPEG, sem DICOM, NIfTI ou metadados clinicos.
- O `inferred_group_id` e uma aproximacao tecnica baseada no nome do arquivo.
- Nao ha tratamento/intervencao nem controle causal nesta etapa; a analise e descritiva e metodologica.
- O outcome preditivo futuro e a classe `Hepatic_Steatosis` versus `Healthy`.


## Setup

Carregamos bibliotecas, caminhos e pequenas funcoes auxiliares. A celula nao altera dados e nao cria arquivos.


In [1]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
REPORT_TABLES = PROJECT_ROOT / "reports" / "tables"
REPORT_FIGURES = PROJECT_ROOT / "reports" / "figures"

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 30)


def read_csv_if_exists(path: Path) -> pd.DataFrame | None:
    if not path.exists():
        print(f"Arquivo nao encontrado: {path}")
        return None
    return pd.read_csv(path)


def read_json_if_exists(path: Path) -> dict | None:
    if not path.exists():
        print(f"Arquivo nao encontrado: {path}")
        return None
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


## Data

Carregamos os indices derivados, se existirem. Caso algum arquivo nao esteja disponivel no ambiente, o notebook continua executando e informa qual script precisa ser rodado localmente.


In [2]:
dataset_index = read_csv_if_exists(DATA_INTERIM / "dataset_index.csv")
group_index = read_csv_if_exists(DATA_INTERIM / "group_index.csv")
split_groups = read_csv_if_exists(DATA_INTERIM / "split_groups.csv")
split_slices = read_csv_if_exists(DATA_INTERIM / "split_slices.csv")

available = {
    "dataset_index": dataset_index is not None,
    "group_index": group_index is not None,
    "split_groups": split_groups is not None,
    "split_slices": split_slices is not None,
}
available


{'dataset_index': True,
 'group_index': True,
 'split_groups': True,
 'split_slices': True}

## Granularidade e chaves

A granularidade esperada de `dataset_index.csv` e uma linha por slice. A granularidade esperada de `group_index.csv` e uma linha por grupo inferido.

As chaves tecnicas de interesse sao `filename`, `inferred_group_id`, `slice_id`, `class_name` e `label`.


In [3]:
if dataset_index is not None:
    print("Linhas em dataset_index:", len(dataset_index))
    print("Arquivos unicos:", dataset_index["file_path"].nunique() if "file_path" in dataset_index else "coluna ausente")
    print("Grupos inferidos:", dataset_index["inferred_group_id"].nunique())
    display(dataset_index[["class_name", "label", "filename", "inferred_group_id", "slice_id", "extension"]].head())

if group_index is not None:
    print("Linhas em group_index:", len(group_index))
    print("Grupos unicos:", group_index["inferred_group_id"].nunique())
    display(group_index.head())


Linhas em dataset_index: 3557
Arquivos unicos: 3557
Grupos inferidos: 225


,class_name,label,filename,inferred_group_id,slice_id,extension
0,Healthy,0,1-img-00004-00080.jpg,1-img-00004,80,.jpg
1,Healthy,0,1-img-00004-00081.jpg,1-img-00004,81,.jpg
2,Healthy,0,1-img-00004-00082.jpg,1-img-00004,82,.jpg
3,Healthy,0,1-img-00004-00083.jpg,1-img-00004,83,.jpg
4,Healthy,0,1-img-00004-00084.jpg,1-img-00004,84,.jpg


Linhas em group_index: 225
Grupos unicos: 225


,inferred_group_id,class_name,label,n_slices,min_slice_id,max_slice_id,total_size_bytes,width_mode,height_mode,n_unreadable
0,1-img-00004,Healthy,0,21,80,100,734218,256,256,0
1,10-img-00011,Hepatic_Steatosis,1,11,19,29,448295,256,256,0
2,100-img-00054,Healthy,0,24,1,24,1146239,256,256,0
3,101-img-00055,Hepatic_Steatosis,1,12,217,228,459643,256,256,0
4,102-img-00056,Hepatic_Steatosis,1,18,217,234,882146,256,256,0


## Distribuicao de classes

Esta checagem e descritiva. Ela mostra quantos slices e quantos grupos inferidos existem por classe.


In [4]:
if dataset_index is not None:
    slices_by_class = (
        dataset_index.groupby("class_name")
        .agg(n_slices=("filename", "count"), n_groups=("inferred_group_id", "nunique"))
        .reset_index()
    )
    display(slices_by_class)

if group_index is not None:
    group_size_summary = (
        group_index.groupby("class_name")["n_slices"]
        .agg(["count", "min", "mean", "median", "max"])
        .round(2)
        .reset_index()
        .rename(columns={"count": "n_groups"})
    )
    display(group_size_summary)


,class_name,n_slices,n_groups
0,Healthy,1611,76
1,Hepatic_Steatosis,1946,149


,class_name,n_groups,min,mean,median,max
0,Healthy,76,11,21.20,22.5,36
1,Hepatic_Steatosis,149,10,13.06,12.0,24


## Validacao do split por grupo

A regra metodologica central e: todos os slices do mesmo `inferred_group_id` devem ficar no mesmo split. O resultado esperado e zero grupos vazando entre splits.


In [5]:
if split_slices is not None:
    split_count_by_group = split_slices.groupby("inferred_group_id")["split"].nunique()
    leaking_groups = split_count_by_group[split_count_by_group > 1]
    print("Grupos com slices em mais de um split:", len(leaking_groups))
    if len(leaking_groups) > 0:
        display(leaking_groups.head(20))
    else:
        display(Markdown("**OK:** nenhum `inferred_group_id` aparece em mais de um split."))

if split_groups is not None:
    split_sets = {
        split_name: set(split_groups.loc[split_groups["split"] == split_name, "inferred_group_id"])
        for split_name in sorted(split_groups["split"].dropna().unique())
    }
    leakage_rows = []
    for left in split_sets:
        for right in split_sets:
            if left < right:
                leakage_rows.append({"comparison": f"{left} vs {right}", "overlap_groups": len(split_sets[left] & split_sets[right])})
    display(pd.DataFrame(leakage_rows))


Grupos com slices em mais de um split: 0


**OK:** nenhum `inferred_group_id` aparece em mais de um split.

,comparison,overlap_groups
0,test vs train,0
1,test vs val,0
2,train vs val,0


## Distribuicao por split

As tabelas abaixo mostram a distribuicao de grupos e slices por split e classe. Elas ajudam a verificar a unidade de analise e possiveis desbalanceamentos.


In [6]:
if split_groups is not None:
    groups_by_split_class = (
        split_groups.groupby(["split", "class_name"])
        .size()
        .reset_index(name="n_groups")
        .sort_values(["split", "class_name"])
    )
    display(groups_by_split_class)

if split_slices is not None:
    slices_by_split_class = (
        split_slices.groupby(["split", "class_name"])
        .size()
        .reset_index(name="n_slices")
        .sort_values(["split", "class_name"])
    )
    display(slices_by_split_class)


,split,class_name,n_groups
0,test,Healthy,12
1,test,Hepatic_Steatosis,23
2,train,Healthy,53
3,train,Hepatic_Steatosis,104
4,val,Healthy,11
5,val,Hepatic_Steatosis,22


,split,class_name,n_slices
0,test,Healthy,260
1,test,Hepatic_Steatosis,300
2,train,Healthy,1095
3,train,Hepatic_Steatosis,1364
4,val,Healthy,256
5,val,Hepatic_Steatosis,282


## Results

A evidencia esperada para esta etapa e: `dataset_index.csv` em granularidade de slice, `group_index.csv` em granularidade de grupo inferido, `split_groups.csv` definindo o split em nivel de grupo e `split_slices.csv` apenas expandindo essa decisao para os slices.

Caso qualquer checagem indique conflito de classe por grupo ou grupo em mais de um split, a modelagem deve parar para investigacao manual.


## Takeaways

O split por `inferred_group_id` reduz o risco de data leakage entre slices correlacionados do mesmo agrupamento. Essa decisao torna a avaliacao menos artificialmente otimista do que um split aleatorio por imagem.

A unidade de entrada do modelo pode ser o slice, mas a avaliacao principal deve incluir agregacao por grupo. O projeto permanece experimental e nao validado clinicamente.
